<a href="https://colab.research.google.com/github/ayush4628/Quora-Duplicate-Question-Pairs/blob/main/Only_bow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/questions.csv")

In [4]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
df.isnull().sum()

,0
id,0
qid1,0
qid2,0
question1,1
question2,2
is_duplicate,0


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
ques_df = df[['question1', 'question2']]
ques_df.head()

,question1,question2
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse

# Ensure ques_df is available in this execution context (assuming it was run previously)
# If running this cell alone, make sure the cell defining ques_df (xI60VPhzeTYp) has been executed.

# Fill NaN values with empty strings for feature extraction
q1_texts = ques_df['question1'].fillna('')
q2_texts = ques_df['question2'].fillna('')

cv = CountVectorizer(max_features=3000) # Max features reduced for memory management

# Fit on the combined vocabulary of both question sets
cv.fit(list(q1_texts) + list(q2_texts))

# Transform question1 and question2 separately into sparse matrices
q1_sparse = cv.transform(q1_texts)
q2_sparse = cv.transform(q2_texts)

# Concatenate the sparse matrices horizontally (axis=1)
# This creates a single sparse feature matrix where each row represents a question pair,
# with features from question1 followed by features from question2.
X_sparse = scipy.sparse.hstack((q1_sparse, q2_sparse))


In [15]:
# X_sparse is now a sparse matrix (scipy.sparse.hstack result)
# You can use X_sparse directly with models that support sparse input (e.g., many sklearn models).
# If you absolutely need a pandas DataFrame, you can convert it, but be mindful of memory if it becomes dense.

# Display the shape of the sparse matrix
print(f"Shape of the sparse feature matrix: {X_sparse.shape}")

# To get an idea of its memory footprint (approximate)
print(f"Approximate memory usage (sparse): {X_sparse.data.nbytes + X_sparse.indptr.nbytes + X_sparse.indices.nbytes} bytes")


Shape of the sparse feature matrix: (404351, 6000)
Approximate memory usage (sparse): 86885028 bytes


In [16]:
X_sparse.shape

(404351, 6000)

In [17]:
X_sparse

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7105635 stored elements and shape (404351, 6000)>

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Define features (X) as the sparse matrix and target (y) as 'is_duplicate'
X = X_sparse
y = df['is_duplicate']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (323480, 6000)
Shape of X_test: (80871, 6000)
Shape of y_train: (323480,)
Shape of y_test: (80871,)


In [19]:
# Initialize and train a RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf.predict(X_test)

# Calculate and print the accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy}")

Accuracy Score: 0.8084232914147222


In [20]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.7468066426778449